# Сегментация разрывных нарушений: CRACKS + U-Net

Этот ноутбук заменяет старый эксперимент с потерянными JPEG-разрезами и ручной разметкой.
Теперь обучение выполняется на открытом наборе **CRACKS (Crowdsourcing Resources for Analysis and Categorization of Key Subsurface faults)**, содержащем реальные сейсмические изображения Netherlands F3 и разметку разломов.

В качестве ground truth используется только каталог **`expert`**, то есть разметка геофизика. После обучения модель можно качественно применить к `TrainingData_Image.segy` из этой курсовой работы.

**Важно:** модель обучается на одном сейсмическом наборе и применяется к другому. Поэтому результат на собственном SEG-Y является примером *domain transfer*, а не количественно подтверждённой точностью: без экспертной разметки собственного куба Dice/IoU для него вычислить нельзя.

Источники:
- CRACKS code: https://github.com/olivesgatech/CRACKS
- CRACKS dataset: https://zenodo.org/records/13926822
- DOI: `10.5281/zenodo.13926822`


## 0. Подготовка окружения

В Colab Google Drive подключается только для доступа к собственному SEG-Y и сохранения весов модели. Сам CRACKS скачивается с Zenodo автоматически (около 115 МБ в двух ZIP-архивах).


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# segyio нужен только для последнего блока с собственным SEG-Y.
!pip -q install segyio pillow


In [ ]:
from __future__ import annotations

import re
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import segyio
import tensorflow as tf
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- CRACKS ---
CRACKS_ROOT = Path('/content/cracks') if Path('/content').exists() else Path('data/cracks')
DOWNLOAD_DIR = CRACKS_ROOT / 'downloads'
IMAGES_DIR = CRACKS_ROOT / 'images'
LABELS_DIR = CRACKS_ROOT / 'labels'

CRACKS_URLS = {
    'images.zip': 'https://zenodo.org/records/13926822/files/images.zip?download=1',
    'Fault segmentations.zip': 'https://zenodo.org/records/13926822/files/Fault%20segmentations.zip?download=1',
}

# --- обучение ---
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 30
INCLUDE_UNCERTAIN = False  # True: добавить зелёные "uncertain fault" пиксели к классу fault

# --- собственный SEG-Y ---
SEGY_PATH = Path('/content/drive/MyDrive/cw_data/TrainingData_Image.segy')
MODEL_DIR = Path('/content/drive/MyDrive/cw_data/models') if Path('/content/drive/MyDrive').exists() else Path('models')
WEIGHTS_PATH = MODEL_DIR / 'cracks_unet.weights.h5'

print('TensorFlow:', tf.__version__)
print('CRACKS root:', CRACKS_ROOT)
print('Model weights:', WEIGHTS_PATH)


## 1. Загрузка CRACKS

Zenodo содержит два архива: 400 сейсмических изображений и набор аннотаций. В архиве аннотаций есть папки разных разметчиков; дальше автоматически выбирается только папка `expert`.


In [ ]:
def download_if_missing(url: str, destination: Path) -> None:
    """Скачивает файл, если он ещё не существует."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f'Already downloaded: {destination.name}')
        return

    print(f'Downloading {destination.name} ...')
    urllib.request.urlretrieve(url, destination)
    print(f'  saved: {destination} ({destination.stat().st_size / 1024**2:.1f} MB)')


def extract_if_needed(archive: Path, destination: Path) -> None:
    """Распаковывает ZIP один раз."""
    marker = destination / '.extracted'
    if marker.exists():
        return

    destination.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {archive.name} ...')
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(destination)
    marker.touch()


for filename, url in CRACKS_URLS.items():
    download_if_missing(url, DOWNLOAD_DIR / filename)

extract_if_needed(DOWNLOAD_DIR / 'images.zip', IMAGES_DIR)
extract_if_needed(DOWNLOAD_DIR / 'Fault segmentations.zip', LABELS_DIR)


## 2. Поиск экспертных масок и сопоставление файлов

Имена файлов в CRACKS кодируют положение разреза в 3D-объёме. Сначала используется нормализованное имя файла; если оно отличается служебным суффиксом (`mask`, `label`, `expert` и т.п.), применяется резервное сопоставление по числовому идентификатору секции.


In [ ]:
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}


def natural_key(text: str):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)', text)]


def normalized_stem(path: Path) -> str:
    stem = path.stem.lower()
    for token in ('ground_truth', 'groundtruth', 'segmentation', 'segmentations',
                  'expert', 'faults', 'fault', 'labels', 'label', 'masks', 'mask', 'gt'):
        stem = stem.replace(token, '')
    return re.sub(r'[^a-z0-9]+', '', stem)


def numeric_signature(path: Path):
    nums = re.findall(r'\d+', path.stem)
    return tuple(int(x) for x in nums) if nums else None


def discover_pairs(images_root: Path, labels_root: Path):
    image_files = sorted(
        [p for p in images_root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS],
        key=lambda p: natural_key(p.name),
    )
    expert_dirs = [p for p in labels_root.rglob('*') if p.is_dir() and p.name.lower() == 'expert']
    if not expert_dirs:
        raise FileNotFoundError('В распакованных аннотациях не найдена папка expert.')

    mask_files = sorted(
        [p for d in expert_dirs for p in d.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS],
        key=lambda p: natural_key(p.name),
    )

    image_by_name = {normalized_stem(p): p for p in image_files}
    mask_by_name = {normalized_stem(p): p for p in mask_files}
    common = sorted(set(image_by_name) & set(mask_by_name), key=natural_key)
    pairs = [(image_by_name[k], mask_by_name[k]) for k in common]

    # Fallback: числовые идентификаторы секций.
    if len(pairs) < max(10, len(mask_files) // 2):
        image_by_num = {}
        mask_by_num = {}
        for p in image_files:
            sig = numeric_signature(p)
            if sig is not None:
                image_by_num.setdefault(sig, []).append(p)
        for p in mask_files:
            sig = numeric_signature(p)
            if sig is not None:
                mask_by_num.setdefault(sig, []).append(p)

        fallback = []
        for sig in sorted(set(image_by_num) & set(mask_by_num)):
            if len(image_by_num[sig]) == 1 and len(mask_by_num[sig]) == 1:
                fallback.append((image_by_num[sig][0], mask_by_num[sig][0]))
        if len(fallback) > len(pairs):
            pairs = fallback

    if not pairs:
        raise RuntimeError('Не удалось сопоставить изображения CRACKS и expert-маски.')

    return sorted(pairs, key=lambda x: natural_key(x[0].name)), expert_dirs


pairs, expert_dirs = discover_pairs(IMAGES_DIR, LABELS_DIR)
print('Expert folders:', expert_dirs)
print('Paired sections:', len(pairs))
for image_path, mask_path in pairs[:5]:
    print(' ', image_path.name, '<->', mask_path.name)


## 3. Интерпретация цветной разметки

В CRACKS цвет маски содержит уверенность разметчика. Для бинарной сегментации:

- **синий** → уверенное наличие разлома;
- **зелёный** → неопределённое наличие разлома;
- остальные пиксели → фон для нашей бинарной задачи.

По умолчанию `INCLUDE_UNCERTAIN = False`, поэтому сеть учится только на уверенных экспертных линиях. Цвет определяется по доминирующему RGB-каналу, а не по одному жёсткому RGB-коду — это устойчивее к сохранению PNG и сглаживанию.


In [ ]:
def inspect_mask_colors(mask_path: Path, top_n: int = 12):
    rgb = np.asarray(Image.open(mask_path).convert('RGB'))
    colors, counts = np.unique(rgb.reshape(-1, 3), axis=0, return_counts=True)
    order = np.argsort(counts)[::-1][:top_n]
    return [(tuple(colors[i]), int(counts[i])) for i in order]


sample_image_path, sample_mask_path = pairs[len(pairs) // 2]
print('Example:', sample_image_path.name, '/', sample_mask_path.name)
print('Most frequent mask colors:')
for color, count in inspect_mask_colors(sample_mask_path):
    print(f'  RGB={color}: {count} px')


In [ ]:
def read_seismic_image(path: Path, size: int = IMG_SIZE) -> np.ndarray:
    """Читает сейсмическое изображение, переводит в grayscale и [0, 1]."""
    image = Image.open(path).convert('L')
    image = image.resize((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return array[..., None]


def read_expert_mask(path: Path, size: int = IMG_SIZE, include_uncertain: bool = False) -> np.ndarray:
    """Преобразует цветную экспертную аннотацию CRACKS в бинарную fault-mask."""
    rgb = np.asarray(Image.open(path).convert('RGB'), dtype=np.int16)
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]

    confident_fault = (b > 80) & (b - r > 20) & (b - g > 20)
    uncertain_fault = (g > 80) & (g - r > 20) & (g - b > 20)

    mask = confident_fault | (uncertain_fault if include_uncertain else False)
    mask_image = Image.fromarray((mask.astype(np.uint8) * 255), mode='L')
    mask_image = mask_image.resize((size, size), Image.Resampling.NEAREST)
    return (np.asarray(mask_image, dtype=np.float32) / 255.0)[..., None]


sample_image = read_seismic_image(sample_image_path)
sample_mask = read_expert_mask(sample_mask_path, include_uncertain=INCLUDE_UNCERTAIN)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample_image[..., 0], cmap='gray', aspect='auto')
axes[0].set_title('CRACKS: seismic section')
axes[1].imshow(sample_mask[..., 0], cmap='gray', vmin=0, vmax=1, aspect='auto')
axes[1].set_title('Expert binary fault mask')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 4. Пространственный train / validation / test split

Соседние сейсмические разрезы сильно похожи. Поэтому случайное перемешивание всех 400 секций перед разделением может привести к утечке пространственно соседней информации и завышенным метрикам.

Здесь пары сначала сортируются по имени секции (имя кодирует положение в объёме), а затем делятся **непрерывными блоками**: 70% / 15% / 15%. Это более строгая проверка, чем случайный split.


In [ ]:
def split_contiguous(items, train_fraction=0.70, val_fraction=0.15):
    n = len(items)
    train_end = int(n * train_fraction)
    val_end = int(n * (train_fraction + val_fraction))
    return items[:train_end], items[train_end:val_end], items[val_end:]


train_pairs, val_pairs, test_pairs = split_contiguous(pairs)
print(f'train={len(train_pairs)}, validation={len(val_pairs)}, test={len(test_pairs)}')
print('train range:', train_pairs[0][0].name, '...', train_pairs[-1][0].name)
print('test range: ', test_pairs[0][0].name, '...', test_pairs[-1][0].name)


In [ ]:
def load_pairs_to_arrays(file_pairs):
    x = np.stack([read_seismic_image(img) for img, _ in file_pairs]).astype(np.float32)
    y = np.stack([
        read_expert_mask(mask, include_uncertain=INCLUDE_UNCERTAIN)
        for _, mask in file_pairs
    ]).astype(np.float32)
    return x, y


X_train, y_train = load_pairs_to_arrays(train_pairs)
X_val, y_val = load_pairs_to_arrays(val_pairs)
X_test, y_test = load_pairs_to_arrays(test_pairs)

fault_fraction = float(y_train.mean())
print('Shapes:', X_train.shape, y_train.shape)
print(f'Fault pixels in train: {fault_fraction:.4%}')


## 5. `tf.data` и аугментация

Для тонких структур разломов важна одинаковая геометрическая трансформация изображения и маски. Используется только горизонтальное отражение; вертикальное отражение не применяется, потому что оно физически переворачивает временную/глубинную ось. Контраст меняется только у входного изображения.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    image = tf.image.random_contrast(image, lower=0.90, upper=1.10)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask


def make_dataset(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)


## 6. Небольшая U-Net

Задача формулируется как **semantic segmentation**: для каждого пикселя предсказывается вероятность принадлежности к разлому.

Разломы занимают малую долю изображения, поэтому одной pixel-wise binary cross-entropy недостаточно. Используется сумма BCE и Dice loss: BCE стабилизирует классификацию пикселей, а Dice непосредственно поощряет перекрытие тонкой маски разлома.


In [ ]:
def conv_block(x, filters: int):
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation('relu')(x)


def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 1)):
    inputs = keras.Input(shape=input_shape)

    c1 = conv_block(inputs, 16)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 32)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 64)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 128)
    p4 = layers.MaxPooling2D()(c4)

    bottleneck = conv_block(p4, 256)
    bottleneck = layers.Dropout(0.30)(bottleneck)

    u4 = layers.UpSampling2D(interpolation='bilinear')(bottleneck)
    u4 = layers.Concatenate()([u4, c4])
    c5 = conv_block(u4, 128)

    u3 = layers.UpSampling2D(interpolation='bilinear')(c5)
    u3 = layers.Concatenate()([u3, c3])
    c6 = conv_block(u3, 64)

    u2 = layers.UpSampling2D(interpolation='bilinear')(c6)
    u2 = layers.Concatenate()([u2, c2])
    c7 = conv_block(u2, 32)

    u1 = layers.UpSampling2D(interpolation='bilinear')(c7)
    u1 = layers.Concatenate()([u1, c1])
    c8 = conv_block(u1, 16)

    outputs = layers.Conv2D(1, 1, activation='sigmoid', name='fault_probability')(c8)
    return keras.Model(inputs, outputs, name='cracks_unet')


def soft_dice(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=(1, 2, 3))
    denominator = tf.reduce_sum(y_true + y_pred, axis=(1, 2, 3))
    return tf.reduce_mean((2.0 * intersection + smooth) / (denominator + smooth))


bce = keras.losses.BinaryCrossentropy()


def bce_dice_loss(y_true, y_pred):
    return bce(y_true, y_pred) + (1.0 - soft_dice(y_true, y_pred))


model = build_unet()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=bce_dice_loss,
    metrics=[soft_dice, keras.metrics.BinaryAccuracy(name='pixel_accuracy')],
)
model.summary()


## 7. Обучение

Лучшие веса сохраняются отдельно (`*.weights.h5`), поэтому модель с пользовательской функцией потерь не требует сложной сериализации. Early stopping возвращает лучший вариант по validation loss.


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=6, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        WEIGHTS_PATH, monitor='val_loss', save_best_only=True,
        save_weights_only=True, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='validation')
axes[0].set_title('BCE + Dice loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['soft_dice'], label='train')
axes[1].plot(history.history['val_soft_dice'], label='validation')
axes[1].set_title('Soft Dice')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()


## 8. Выбор порога и тестовые метрики

Порог бинаризации не выбирается по test set. Сначала на validation set перебирается диапазон порогов и выбирается лучший Dice, после чего этот единственный порог применяется к отложенному test set.


In [ ]:
def binary_metrics(y_true, y_prob, threshold=0.5, eps=1e-8):
    true = y_true.astype(bool).ravel()
    pred = (y_prob >= threshold).ravel()

    tp = np.logical_and(true, pred).sum()
    fp = np.logical_and(~true, pred).sum()
    fn = np.logical_and(true, ~pred).sum()

    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return {'dice': dice, 'iou': iou, 'precision': precision, 'recall': recall}


val_prob = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
thresholds = np.arange(0.20, 0.81, 0.05)
val_scores = [binary_metrics(y_val, val_prob, t)['dice'] for t in thresholds]
BEST_THRESHOLD = float(thresholds[int(np.argmax(val_scores))])

plt.figure(figsize=(7, 4))
plt.plot(thresholds, val_scores, marker='o')
plt.axvline(BEST_THRESHOLD, linestyle='--')
plt.xlabel('Probability threshold')
plt.ylabel('Validation Dice')
plt.title(f'Chosen threshold = {BEST_THRESHOLD:.2f}')
plt.show()


test_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
test_metrics = binary_metrics(y_test, test_prob, BEST_THRESHOLD)
print('TEST METRICS')
for name, value in test_metrics.items():
    print(f'{name:>10}: {value:.4f}')


In [ ]:
def show_predictions(x, y, prob, threshold, indices=(0, 1, 2)):
    indices = [i for i in indices if i < len(x)]
    fig, axes = plt.subplots(len(indices), 4, figsize=(14, 4 * len(indices)))
    axes = np.atleast_2d(axes)

    for row, idx in enumerate(indices):
        axes[row, 0].imshow(x[idx, ..., 0], cmap='gray', aspect='auto')
        axes[row, 0].set_title('Seismic image')

        axes[row, 1].imshow(y[idx, ..., 0], cmap='gray', vmin=0, vmax=1, aspect='auto')
        axes[row, 1].set_title('Expert mask')

        axes[row, 2].imshow(prob[idx, ..., 0], cmap='viridis', vmin=0, vmax=1, aspect='auto')
        axes[row, 2].set_title('Fault probability')

        axes[row, 3].imshow(x[idx, ..., 0], cmap='gray', aspect='auto')
        axes[row, 3].contour(
            prob[idx, ..., 0] >= threshold,
            levels=[0.5], linewidths=1.0,
        )
        axes[row, 3].set_title('Prediction overlay')

        for col in range(4):
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()


show_predictions(X_test, y_test, test_prob, BEST_THRESHOLD, indices=(0, len(X_test)//2, len(X_test)-1))


## 9. Применение к собственному `TrainingData_Image.segy`

Этот блок не участвует в обучении и не выдаёт количественную оценку качества. Он показывает перенос модели, обученной на Netherlands F3/CRACKS, на другой сейсмический набор.

Собственный SEG-Y преобразуется в grayscale по амплитуде с percentile clipping. Затем весь выбранный 2D-фрагмент временно приводится к размеру входа U-Net, а карта вероятностей масштабируется обратно для визуализации.

Это сознательно **качественный baseline**. Для серьёзного переноса на свой куб нужны хотя бы несколько экспертно размеченных разрезов для fine-tuning и независимой проверки.


In [ ]:
def load_segy_fragment(path: Path, start_trace: int, n_traces: int) -> np.ndarray:
    """Возвращает сейсмический фрагмент формы (samples, traces)."""
    with segyio.open(str(path), 'r', ignore_geometry=True) as segy:
        total = len(segy.trace)
        n_samples = len(segy.samples)
        start = max(0, min(start_trace, total - 1))
        stop = min(start + n_traces, total)
        if stop <= start:
            raise ValueError('Пустой диапазон трасс.')
        section = np.stack([np.asarray(segy.trace[i], dtype=np.float32) for i in range(start, stop)], axis=1)
    return section


def seismic_amplitude_to_image(section: np.ndarray, clip_percentile: float = 99.0) -> np.ndarray:
    """Симметрично клиппирует амплитуды и переводит их в [0, 1]."""
    scale = float(np.percentile(np.abs(section), clip_percentile))
    if scale <= 0:
        return np.zeros_like(section, dtype=np.float32)
    clipped = np.clip(section, -scale, scale)
    return ((clipped / scale) + 1.0).astype(np.float32) / 2.0


def predict_section(model, section_image: np.ndarray, threshold: float):
    h, w = section_image.shape
    small = Image.fromarray((section_image * 255).astype(np.uint8), mode='L')
    small = small.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    x = np.asarray(small, dtype=np.float32)[None, ..., None] / 255.0

    prob_small = model.predict(x, verbose=0)[0, ..., 0]
    prob_image = Image.fromarray(prob_small.astype(np.float32), mode='F')
    prob_image = prob_image.resize((w, h), Image.Resampling.BILINEAR)
    probability = np.asarray(prob_image, dtype=np.float32)
    return probability, probability >= threshold


In [ ]:
if SEGY_PATH.exists():
    with segyio.open(str(SEGY_PATH), 'r', ignore_geometry=True) as segy:
        total_traces = len(segy.trace)

    # В исходном эксперименте наиболее наглядный разлом наблюдался в конце набора трасс.
    N_TRACES = 700
    START_TRACE = max(0, total_traces - N_TRACES)

    section = load_segy_fragment(SEGY_PATH, START_TRACE, N_TRACES)
    section_image = seismic_amplitude_to_image(section)
    probability, predicted_mask = predict_section(model, section_image, BEST_THRESHOLD)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(section_image, cmap='gray', aspect='auto')
    axes[0].set_title('Own SEG-Y: seismic section')

    im = axes[1].imshow(probability, cmap='viridis', vmin=0, vmax=1, aspect='auto')
    axes[1].set_title('Transferred U-Net: fault probability')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(section_image, cmap='gray', aspect='auto')
    axes[2].contour(predicted_mask, levels=[0.5], linewidths=0.9)
    axes[2].set_title('Predicted faults (qualitative transfer)')

    for ax in axes:
        ax.set_xlabel('Trace')
        ax.set_ylabel('Sample')
    plt.tight_layout()
    plt.show()
else:
    print(f'SEG-Y not found: {SEGY_PATH}')
    print('CRACKS training/evaluation works without it; set SEGY_PATH to run transfer inference.')


## Выводы и ограничения

1. В отличие от старого CV-ноутбука, здесь есть реальный внешний ground truth и отдельные train/validation/test части.
2. U-Net решает задачу именно **пиксельной сегментации разломов**, а не поиска произвольных прямых линий.
3. Для CRACKS можно вычислить Dice/IoU/precision/recall, потому что известна экспертная разметка.
4. Для собственного SEG-Y без разметки можно показывать только качественный результат переноса модели.
5. CRACKS и собственный куб могут отличаться динамическим диапазоном, частотным составом, масштабом, ориентацией и стилем визуализации. Это **domain shift** и основное ограничение переноса.
6. Если понадобится повысить качество на собственном кубе, следующий корректный шаг — вручную разметить небольшой набор его разрезов и выполнить fine-tuning, сохранив отдельные разрезы только для финального теста.
